In [13]:
import pandas as pd
import cupy as cp
import cudf
import cuml
import torch
import gc
import time
import matplotlib.pyplot as plt
import seaborn as sns
from cuml.cluster import DBSCAN
from sklearn.decomposition import PCA

class Clustering(object):

    def __init__(self, eps: float, min_samples: int, 
                 metric: str, algorithm: str, 
                 verbose: bool,
                 max_mbytes_per_batch: int,
                 output_type: str, calc_core_sample_indices: bool,
                 plotting_x: str, plotting_y: str,):
        self.eps = eps
        self.min_samples = min_samples
        self.metric = metric
        self.algorithm = algorithm
        self.verbose = verbose
        self.max_mbytes_per_batch = max_mbytes_per_batch
        self.output_type = output_type
        self.calc_core_sample_indices = calc_core_sample_indices
        self.plotting_x = plotting_x
        self.plotting_y = plotting_y

    def df_read(self):
        global df_pandas_global
        
        df_pandas = pd.read_csv('household_power_consumption_not_null.csv', parse_dates=[['Date', 'Time']],
                                date_format = {'Date': '%d/%m/%Y', 'Time': '%H:%M:%S'},
                                dayfirst = True)
        df_pandas_global = df_pandas
    
    def to_cudf(self):
        global df_cudf_global
        
        df_cudf = cudf.DataFrame(df_pandas_global).copy()
        df_cudf_global = df_cudf
        
    def converting(self):
        global df_cudf_converted_global
        
        df_cudf_converted_global = df_cudf_global.astype(float).copy()

    def DBSCAN(self):
        global DBSCAN_global
        global labels_global
        global cluster_centers_global

        DBSCAN_float = cuml.cluster.DBSCAN(eps=self.eps, min_samples=self.min_samples, metric=self.metric, 
                                                  algorithm=self.algorithm, verbose=self.verbose, 
                                                  max_mbytes_per_batch=self.max_mbytes_per_batch, 
                                                  output_type=self.output_type, 
                                                  calc_core_sample_indices=self.calc_core_sample_indices)
        DBSCAN = DBSCAN_float.fit(df_cudf_converted_global)
        DBSCAN_global = DBSCAN

        labels = DBSCAN_float.labels_
        labels_global = cudf.DataFrame(labels, columns=['Labels'])

    def labels_concat(self):
        global df_concat_global
        global df_concat_pandas_global

        df_concat = cudf.concat([df_cudf_converted_global, labels_global], axis=1)
        df_concat_global = df_concat

        df_concat_pandas = df_concat.to_pandas()
        df_concat_pandas_global = df_concat_pandas

    def plotting(self):


        x=df_concat_pandas_global[self.plotting_x]
        y=df_concat_pandas_global[self.plotting_y]
        Cluster = df_concat_pandas_global['Labels']

        fig = plt.figure()
        ax = fig.add_subplot(111)
        scatter = ax.scatter(x,y,c=Cluster,s=50)
        plt.colorbar(scatter)
        plt.xlabel(self.plotting_x)
        plt.ylabel(self.plotting_y)

        fig.show()

        # Reduce to 2 dimensions
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(df_concat_pandas_global)

        # Plot the reduced data
        plt.figure(figsize=(8, 6))
        scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df_concat_pandas_global['Labels'], cmap='coolwarm', s=50)

        plt.title("Clustering Labels visualized via PCA")
        plt.show()
        
        
    def main(self):
        self.df_read()
        self.to_cudf()
        self.converting()
        st = time.time()
        self.DBSCAN()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')
        st = time.time()
        self.labels_concat()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')
        st = time.time()
        self.plotting()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')
        

In [14]:
#main_clustering = Clustering(0.5, 5, 'euclidean', 'brute', 
 #                            False, None, None, True, 'Global_active_power', 'Voltage')

In [ ]:
#main_clustering.main()

/tmp/ipykernel_24033/2759924248.py:35: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  df_pandas = pd.read_csv('household_power_consumption_not_null.csv', parse_dates=[['Date', 'Time']],
